In [6]:
import requests
import json
import os
from dotenv import load_dotenv

load_dotenv()

# URLs corrigées
LOGIN_URL = "https://registre-national-entreprises.inpi.fr/api/sso/login"
# Correction ici : l'endpoint de recherche est souvent direct ou nécessite 'search' différemment
SEARCH_URL = "https://registre-national-entreprises.inpi.fr/api/companies" 
SAVE_DIR = "raw_data"

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

def get_auth_token():
    payload = {
        "username": os.getenv("INPI_USERNAME"),
        "password": os.getenv("INPI_PASSWORD")
    }
    try:
        response = requests.post(LOGIN_URL, json=payload)
        response.raise_for_status()
        return response.json().get("token")
    except requests.exceptions.RequestException as e:
        print(f"Erreur d'authentification : {e}")
        return None

def fetch_data(token, page=1):
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json"
    }
    
    # On utilise des paramètres de requête (Params) plutôt qu'un Body JSON pour une recherche simple
    # Le code NAF 6201Z correspond au secteur informatique [cite: 37]
    params = {
        "page": page,
        "pageSize": 20,
        "activitePrincipale": "62.01Z"
    }

    try:
        # Utilisation de GET au lieu de POST pour la recherche standard
        response = requests.get(SEARCH_URL, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()
        
        file_path = os.path.join(SAVE_DIR, f"data_page_{page}.json")
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        
        print(f"Succès ! Données enregistrées dans {file_path}")
        return data
    except requests.exceptions.RequestException as e:
        print(f"Erreur lors de l'extraction : {e}")
        # Si le 404 persiste, l'API demande peut-être un SIREN spécifique
        return None

token_valide = get_auth_token()
if token_valide:
    fetch_data(token_valide, page=1)

Succès ! Données enregistrées dans raw_data\data_page_1.json
